# Zaid Hybrid IDS — Step 5: Dst Port Ablation and Sensitivity
This notebook evaluates whether the validation performance of the selected Random Forest detector depends on Dst Port. It compares the existing full model, the same model retrained without Dst Port, and the full model under a deterministic shuffled-port sensitivity test. The test partition is not loaded or used.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
from time import perf_counter
import json
import platform
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score
)

PROJECT_DIR = Path('/content/drive/MyDrive/Zaid_Hybrid_IDS')
SPLIT_DIR = PROJECT_DIR / 'data' / 'splits'
RESULT_DIR = PROJECT_DIR / 'results'
MODEL_DIR = PROJECT_DIR / 'models'
SEED = 20260829

TRAIN_PATH = SPLIT_DIR / 'train.parquet'
VALIDATION_PATH = SPLIT_DIR / 'validation.parquet'
FEATURE_MANIFEST_PATH = RESULT_DIR / 'feature_sets_manifest.json'
BASELINE_PREDICTIONS_PATH = RESULT_DIR / 'baseline_validation_predictions.parquet'
FULL_MODEL_PATH = MODEL_DIR / 'random_forest.joblib'

for path in [TRAIN_PATH, VALIDATION_PATH, FEATURE_MANIFEST_PATH, BASELINE_PREDICTIONS_PATH, FULL_MODEL_PATH]:
    assert path.exists(), f'Missing required file: {path}'

print('Loading train and validation only...')
train_df = pd.read_parquet(TRAIN_PATH)
validation_df = pd.read_parquet(VALIDATION_PATH)
baseline_predictions = pd.read_parquet(BASELINE_PREDICTIONS_PATH)
with open(FEATURE_MANIFEST_PATH, 'r', encoding='utf-8') as file_handle:
    feature_manifest = json.load(file_handle)
full_model = joblib.load(FULL_MODEL_PATH)

assert len(train_df) == 731315
assert len(validation_df) == 156714
assert len(baseline_predictions) == len(validation_df)
print('Train:', train_df.shape)
print('Validation:', validation_df.shape)
print('Test partition loaded: False')

Loading train and validation only...
Train: (731315, 81)
Validation: (156714, 81)
Test partition loaded: False


In [3]:
FULL_FEATURES = feature_manifest['full_feature_set']
NO_PORT_FEATURES = feature_manifest['no_dst_port_feature_set']
assert len(FULL_FEATURES) == 61
assert len(NO_PORT_FEATURES) == 60
assert 'Dst Port' in FULL_FEATURES and 'Dst Port' not in NO_PORT_FEATURES

X_train_no_port = train_df[NO_PORT_FEATURES].astype(np.float32)
X_validation_no_port = validation_df[NO_PORT_FEATURES].astype(np.float32)
X_validation_full = validation_df[FULL_FEATURES].astype(np.float32)
y_train = train_df['Label'].astype(str).ne('Benign').astype(np.int8).to_numpy()
y_validation_original = validation_df['Label'].astype(str).reset_index(drop=True)
y_validation = y_validation_original.ne('Benign').astype(np.int8).to_numpy()

assert np.array_equal(y_validation, baseline_predictions['binary_label'].to_numpy())
assert np.isfinite(X_train_no_port.to_numpy(copy=False)).all()
assert np.isfinite(X_validation_no_port.to_numpy(copy=False)).all()

print('Full features:', len(FULL_FEATURES))
print('No-Dst-Port features:', len(NO_PORT_FEATURES))
print('Ablation data preparation: PASSED')

Full features: 61
No-Dst-Port features: 60
Ablation data preparation: PASSED


In [4]:
def evaluate_probability(scenario, probability, inference_seconds, fit_seconds=None, model_size_mb=None):
    probability = np.asarray(probability, dtype=np.float64)
    assert np.isfinite(probability).all()
    prediction = (probability >= 0.5).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_validation, prediction, labels=[0, 1]).ravel()
    ftp_mask = y_validation_original.eq('FTP-BruteForce').to_numpy()
    ssh_mask = y_validation_original.eq('SSH-Bruteforce').to_numpy()
    return {
        'scenario': scenario,
        'threshold': 0.5,
        'accuracy': accuracy_score(y_validation, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_validation, prediction),
        'precision': precision_score(y_validation, prediction, zero_division=0),
        'recall': recall_score(y_validation, prediction, zero_division=0),
        'f1': f1_score(y_validation, prediction, zero_division=0),
        'macro_f1': f1_score(y_validation, prediction, average='macro'),
        'pr_auc': average_precision_score(y_validation, probability),
        'roc_auc': roc_auc_score(y_validation, probability),
        'brier_score': brier_score_loss(y_validation, probability),
        'false_positive_rate': fp / (fp + tn) if (fp + tn) else 0.0,
        'ftp_attack_recall': float(prediction[ftp_mask].mean()),
        'ssh_attack_recall': float(prediction[ssh_mask].mean()),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'fit_seconds': fit_seconds,
        'validation_inference_seconds': inference_seconds,
        'batch_mean_microseconds_per_flow': inference_seconds / len(y_validation) * 1e6,
        'batch_throughput_flows_per_second': len(y_validation) / inference_seconds,
        'model_size_mb': model_size_mb
    }, prediction

full_probability = baseline_predictions['random_forest_probability'].to_numpy()
full_prediction = baseline_predictions['random_forest_prediction'].to_numpy().astype(np.int8)
full_model_size_mb = FULL_MODEL_PATH.stat().st_size / (1024 ** 2)
full_metrics, full_prediction_check = evaluate_probability(
    'full_model_existing', full_probability, inference_seconds=1.078609,
    fit_seconds=161.414875, model_size_mb=full_model_size_mb
)
assert np.array_equal(full_prediction, full_prediction_check)
print('Evaluation function and full-model reference: READY')

Evaluation function and full-model reference: READY


In [5]:
print('Training the same Random Forest without Dst Port...')
no_port_model = RandomForestClassifier(
    n_estimators=150, max_depth=20, min_samples_leaf=2,
    max_features='sqrt', class_weight='balanced_subsample',
    n_jobs=-1, random_state=SEED
)
fit_start = perf_counter()
no_port_model.fit(X_train_no_port, y_train)
no_port_fit_seconds = perf_counter() - fit_start
inference_start = perf_counter()
no_port_probability = no_port_model.predict_proba(X_validation_no_port)[:, 1]
no_port_inference_seconds = perf_counter() - inference_start

NO_PORT_MODEL_PATH = MODEL_DIR / 'random_forest_no_dst_port.joblib'
joblib.dump(no_port_model, NO_PORT_MODEL_PATH, compress=3)
no_port_model_size_mb = NO_PORT_MODEL_PATH.stat().st_size / (1024 ** 2)
no_port_metrics, no_port_prediction = evaluate_probability(
    'retrained_without_dst_port', no_port_probability,
    inference_seconds=no_port_inference_seconds,
    fit_seconds=no_port_fit_seconds, model_size_mb=no_port_model_size_mb
)
print(pd.Series(no_port_metrics).to_string())
print('Saved model:', NO_PORT_MODEL_PATH)

Training the same Random Forest without Dst Port...
scenario                             retrained_without_dst_port
threshold                                                   0.5
accuracy                                               0.999968
balanced_accuracy                                      0.999967
precision                                              0.999948
recall                                                 0.999965
f1                                                     0.999956
macro_f1                                               0.999966
pr_auc                                                      1.0
roc_auc                                                     1.0
brier_score                                            0.000021
false_positive_rate                                     0.00003
ftp_attack_recall                                           1.0
ssh_attack_recall                                      0.999929
tn                                                  

In [6]:
print('Running shuffled-Dst-Port sensitivity test on the existing full model...')
rng = np.random.default_rng(SEED)
X_validation_shuffled_port = X_validation_full.copy()
X_validation_shuffled_port['Dst Port'] = rng.permutation(
    X_validation_shuffled_port['Dst Port'].to_numpy()
)
inference_start = perf_counter()
shuffled_port_probability = full_model.predict_proba(X_validation_shuffled_port)[:, 1]
shuffled_port_inference_seconds = perf_counter() - inference_start
shuffled_metrics, shuffled_prediction = evaluate_probability(
    'full_model_shuffled_dst_port', shuffled_port_probability,
    inference_seconds=shuffled_port_inference_seconds,
    fit_seconds=None, model_size_mb=full_model_size_mb
)
print(pd.Series(shuffled_metrics).to_string())

Running shuffled-Dst-Port sensitivity test on the existing full model...
scenario                             full_model_shuffled_dst_port
threshold                                                     0.5
accuracy                                                 0.828994
balanced_accuracy                                        0.765532
precision                                                0.999835
recall                                                   0.531114
f1                                                       0.693722
macro_f1                                                 0.787553
pr_auc                                                   0.999267
roc_auc                                                  0.999637
brier_score                                              0.077913
false_positive_rate                                       0.00005
ftp_attack_recall                                        0.377694
ssh_attack_recall                                        0.689257
tn 

In [7]:
comparison = pd.DataFrame([full_metrics, no_port_metrics, shuffled_metrics])
full_macro_f1 = float(comparison.loc[comparison['scenario'].eq('full_model_existing'), 'macro_f1'].iloc[0])
no_port_macro_f1 = float(comparison.loc[comparison['scenario'].eq('retrained_without_dst_port'), 'macro_f1'].iloc[0])
shuffled_macro_f1 = float(comparison.loc[comparison['scenario'].eq('full_model_shuffled_dst_port'), 'macro_f1'].iloc[0])
comparison['macro_f1_delta_vs_full'] = comparison['macro_f1'] - full_macro_f1

predictions = pd.DataFrame({
    'validation_row': np.arange(len(validation_df), dtype=np.int64),
    'flow_group_id': validation_df['flow_group_id'].to_numpy(),
    'original_label': y_validation_original,
    'binary_label': y_validation,
    'full_probability': full_probability.astype(np.float32),
    'full_prediction': full_prediction,
    'no_port_probability': no_port_probability.astype(np.float32),
    'no_port_prediction': no_port_prediction,
    'shuffled_port_probability': shuffled_port_probability.astype(np.float32),
    'shuffled_port_prediction': shuffled_prediction
})

full_importance = pd.DataFrame({
    'feature': FULL_FEATURES,
    'full_model_importance': full_model.feature_importances_
})
no_port_importance = pd.DataFrame({
    'feature': NO_PORT_FEATURES,
    'no_port_model_importance': no_port_model.feature_importances_
})
importance = full_importance.merge(no_port_importance, on='feature', how='outer')
importance['full_model_rank'] = importance['full_model_importance'].rank(ascending=False, method='min')
importance['no_port_model_rank'] = importance['no_port_model_importance'].rank(ascending=False, method='min')
importance = importance.sort_values('full_model_importance', ascending=False, na_position='last')
dst_port_row = importance.loc[importance['feature'].eq('Dst Port')].iloc[0]

METRICS_PATH = RESULT_DIR / 'dst_port_ablation_validation_metrics.csv'
PREDICTIONS_PATH = RESULT_DIR / 'dst_port_ablation_validation_predictions.parquet'
IMPORTANCE_PATH = RESULT_DIR / 'dst_port_ablation_feature_importance.csv'
MANIFEST_PATH = RESULT_DIR / 'dst_port_ablation_manifest.json'
comparison.to_csv(METRICS_PATH, index=False)
predictions.to_parquet(PREDICTIONS_PATH, index=False, compression='zstd')
importance.to_csv(IMPORTANCE_PATH, index=False)

manifest = {
    'stage': 'dst_port_ablation_validation',
    'seed': SEED,
    'selected_architecture': 'RandomForestClassifier',
    'full_feature_count': len(FULL_FEATURES),
    'no_dst_port_feature_count': len(NO_PORT_FEATURES),
    'test_partition_loaded_or_used': False,
    'full_macro_f1': full_macro_f1,
    'no_port_macro_f1': no_port_macro_f1,
    'no_port_macro_f1_delta': no_port_macro_f1 - full_macro_f1,
    'shuffled_port_macro_f1': shuffled_macro_f1,
    'shuffled_port_macro_f1_delta': shuffled_macro_f1 - full_macro_f1,
    'dst_port_full_model_importance': float(dst_port_row['full_model_importance']),
    'dst_port_full_model_rank': int(dst_port_row['full_model_rank']),
    'sensitivity_note': 'Shuffling Dst Port creates a diagnostic stress test and may create unrealistic feature combinations; it is not a deployment estimate',
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'python_version': sys.version,
    'platform': platform.platform()
}
with open(MANIFEST_PATH, 'w', encoding='utf-8') as file_handle:
    json.dump(manifest, file_handle, ensure_ascii=False, indent=2)

display_columns = [
    'scenario', 'macro_f1', 'f1', 'precision', 'recall', 'pr_auc',
    'false_positive_rate', 'fp', 'fn', 'macro_f1_delta_vs_full',
    'batch_mean_microseconds_per_flow', 'model_size_mb'
]
print(comparison[display_columns].to_string(index=False))
print('\nDst Port importance:', float(dst_port_row['full_model_importance']))
print('Dst Port rank:', int(dst_port_row['full_model_rank']))
print('Saved:', METRICS_PATH)
print('Saved:', PREDICTIONS_PATH)
print('Saved:', IMPORTANCE_PATH)
print('Saved:', MANIFEST_PATH)
print('DST PORT ABLATION PASSED: validation comparison saved; test remains untouched.')

                    scenario  macro_f1       f1  precision   recall   pr_auc  false_positive_rate  fp    fn  macro_f1_delta_vs_full  batch_mean_microseconds_per_flow  model_size_mb
         full_model_existing  0.999993 0.999991   0.999983 1.000000 1.000000              0.00001   1     0                0.000000                          6.882659       0.304957
  retrained_without_dst_port  0.999966 0.999956   0.999948 0.999965 1.000000              0.00003   3     2               -0.000028                          5.717969       0.525250
full_model_shuffled_dst_port  0.787553 0.693722   0.999835 0.531114 0.999267              0.00005   5 26794               -0.212440                          4.717021       0.304957

Dst Port importance: 0.16121566159843717
Dst Port rank: 2
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/dst_port_ablation_validation_metrics.csv
Saved: /content/drive/MyDrive/Zaid_Hybrid_IDS/results/dst_port_ablation_validation_predictions.parquet
Saved: /content/dri